In [ ]:
import os
import subprocess
import re
import csv
from concurrent.futures import ThreadPoolExecutor
from openbabel import openbabel
from openpyxl import Workbook
import time
import traceback
import sys

In [ ]:




os.environ['PATH'] = '/home/fwtop/apps/openmpi/bin:' + os.environ.get('PATH', '')

os.environ['LD_LIBRARY_PATH'] = '/home/fwtop/apps/openmpi/lib:' + os.environ.get('LD_LIBRARY_PATH', '')

os.environ['OMPI_ALLOW_RUN_AS_ROOT'] = '1'
os.environ['OMPI_ALLOW_RUN_AS_ROOT_CONFIRM'] = '1'

In [ ]:

MAX_TASKS = 2

In [ ]:

os.makedirs('dimer_gas/opt+freq/success', exist_ok=True)
os.makedirs('dimer_gas/opt+freq/failure', exist_ok=True)

In [ ]:

base_dir = 'dimer_gas/opt+freq'
success_dir = os.path.join(base_dir, 'success')
failure_dir = os.path.join(base_dir, 'failure')

In [ ]:
def run_orca_gas_optfreq(file, success_dir, failure_dir):
    
    
    base_name = os.path.splitext(file)[0]
    output_file = base_name + '.out'
    opt_file = base_name + '.opt'
    trj_file = base_name + '_trj' + '.xyz'
    xyz_file = base_name + '.xyz'
    property_file = base_name + '.property' + '.txt'
    gbw_file = base_name + '.gbw'
    engrad_file = base_name + '.engrad'
    densitiesinfo_file = base_name + '.densitiesinfo'
    densities_file = base_name + '.densities'
    hess_file = base_name + '.hess'
    bibtex_file = base_name + '.bibtex'

    try:
        
        with open(output_file, 'w') as out:
            subprocess.run(['/home/public/orca_6_0_1_linux_x86-64_shared_openmpi416_avx2/orca', file], stdout=out, check=True)

        
        with open(output_file, 'r') as f:
            content = f.read()
            if 'ORCA TERMINATED NORMALLY' in content:
                
                files_to_move = [
                    file, output_file, opt_file, trj_file, xyz_file,
                    property_file, gbw_file, engrad_file, densitiesinfo_file,
                    densities_file, hess_file, bibtex_file
                ]
                for f in files_to_move:
                    if os.path.exists(f):
                        os.rename(f, os.path.join(success_dir, os.path.basename(f)))
                
                new_output_path = os.path.join(success_dir, os.path.basename(output_file))
                return new_output_path, True
            else:
                
                files_to_move = [
                    file, output_file, opt_file, trj_file, xyz_file,
                    property_file, gbw_file, engrad_file, densitiesinfo_file,
                    densities_file, hess_file, bibtex_file
                ]
                for f in files_to_move:
                    if os.path.exists(f):
                        os.rename(f, os.path.join(failure_dir, os.path.basename(f)))
                return None, False

    except subprocess.CalledProcessError as e:
        
        os.rename(file, os.path.join(failure_dir, os.path.basename(file)))
        if os.path.exists(output_file):
            os.rename(output_file, os.path.join(failure_dir, os.path.basename(output_file)))
        print(f"Error running ORCA for file {file}: {str(e)}")
        print(traceback.format_exc())
        return None, False

In [ ]:

def main():
    
    
    start_time = time.time() 
    
    
    inp_files = [f for f in os.listdir('.') if os.path.isfile(f) and f.endswith('.inp')]
    
    
    with ThreadPoolExecutor(max_workers=MAX_TASKS) as executor:
        
        futures = [executor.submit(run_orca_gas_optfreq, file, success_dir, failure_dir) for file in inp_files] 
        
        results = [f.result() for f in futures]
    
    
    success_results = [result for result in results if result[1]]
    
    
    end_time = time.time()
    
    
    elapsed_time = end_time - start_time
    print(f"The code ran for {elapsed_time} seconds.")



In [ ]:
if __name__ == '__main__':
    main()